# 4.21 Tarea Hogar 03 - Grid Search Arboles Azarosos (32 arbolitos)

Optimizacion simultanea de `feature_fraction`, `cp`, `maxdepth`, `minsplit`, `minbucket`. Basado en `z420_ArbolesAzarosos.ipynb`.

Reportar en la Planilla Colaborativa, hoja **C4 - Grid Search ArbolesAzarosos**.

## Sobre los hiperparametros

- **feature_fraction**: fraccion de columnas que usa cada arbol del ensemble. Bajarlo reduce la correlacion entre arboles y mejora la generalizacion.
- **maxdepth**: profundidad maxima del arbol. rpart no admite mas de 30 aunque se pida.
- **cp**: umbral minimo de mejora para aceptar un split (rpart, seccion 12.2 del libro). El libro plantea probar empiricamente si un cp negativo puede ser optimo para la ganancia, contra la teoria clasica que lo restringe a positivo.
- **minsplit**: minimo de registros en un nodo para intentar dividirlo.
- **minbucket**: minimo de registros en una hoja. Por defecto rpart lo pone en minsplit/3. Con minbucket=1 el arbol memoriza clientes individuales.

Un grid completo con 2 valores por parametro (32 combinaciones) y 32 arboles ya superaba las 12 hs segun lo reportado en el grupo. Con 5 hiperparametros no es viable, asi que va random search en dos etapas: exploracion barata con pocos arboles, y confirmacion de los mejores candidatos con el ensemble completo de 32.

## Paso 1: ambiente Python3 (una sola vez)

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

RUTA_DMEYF="/content/.drive/My Drive/Capacitacion/Maestria Data Minning/09-Aplicaciones_mineria_datos_en_economia/dmeyf"

mkdir -p "$RUTA_DMEYF"
mkdir -p "/content/buckets"
ln -sfn "$RUTA_DMEYF" /content/buckets/b1

mkdir -p ~/.kaggle
cp "$RUTA_DMEYF/kaggle/kaggle.json" ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget "$url_origen""$archivo" -O "$carpeta_destino""$archivo"
  fi

  if ! test -f "/content/datasets/""$archivo"; then
    cp "$carpeta_destino""$archivo" "/content/datasets/""$archivo"
  fi
}

descargar "dataset_pequeno.csv"

## Paso 2: reconstruccion del ambiente en R (correr siempre)

In [ ]:
RUTA_DMEYF <- "/content/.drive/My Drive/Capacitacion/Maestria Data Minning/09-Aplicaciones_mineria_datos_en_economia/dmeyf"

system(paste0(
  "python3 -c \"from google.colab import drive; drive.mount('/content/.drive')\""
), intern = TRUE)

system("mkdir -p /content/buckets", intern = TRUE)
system(paste0('ln -sfn "', RUTA_DMEYF, '" /content/buckets/b1'), intern = TRUE)

system("mkdir -p ~/.kaggle", intern = TRUE)
system("cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle", intern = TRUE)
system("chmod 600 ~/.kaggle/kaggle.json", intern = TRUE)

system("mkdir -p /content/datasets", intern = TRUE)
system("cp /content/buckets/b1/datasets/dataset_pequeno.csv /content/datasets/dataset_pequeno.csv", intern = TRUE)

system('ls -la "/content/datasets/dataset_pequeno.csv"', intern = TRUE)
system('ls -la ~/.kaggle/kaggle.json', intern = TRUE)

In [ ]:
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

require("data.table")
require("rpart")
require("pROC")

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- "exp4210_GridSearch_ArbolesAzarosos"
dir.create(experimento, showWarnings = FALSE)
setwd(paste0("/content/buckets/b1/exp/", experimento))
getwd()

## Paso 3: dataset y holdout de validacion

In [ ]:
dataset <- fread("/content/datasets/dataset_pequeno.csv")

dtrain  <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]
dfuture[, clase_ternaria := NA]

campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

cat("dtrain:", nrow(dtrain), " | campos:", length(campos_buenos), "\n")

El holdout 80/20 es solo para comparar combinaciones por AUC sin gastar las subidas de Kaggle. El modelo final se reentrena con dtrain completo.

In [ ]:
set.seed(100043)

n_total <- nrow(dtrain)
n_train <- as.integer(n_total * 0.8)
indices <- sample(1:n_total)

dtrain_fit <- dtrain[indices[1:n_train]]
dvalid     <- dtrain[indices[(n_train + 1):n_total]]

dvalid[, clase_binaria := ifelse(clase_ternaria == "BAJA+2", 1, 0)]

cat("fit:", nrow(dtrain_fit), " valid:", nrow(dvalid), " tasa BAJA+2:", round(mean(dvalid$clase_binaria), 4), "\n")

## Paso 4: espacio de busqueda

In [ ]:
generar_combinacion <- function() {
  feature_fraction <- round(runif(1, 0.2, 0.8), 2)
  cp <- sample(c(-1, 0, 0.0001, 0.0005, 0.001, 0.005, 0.01), 1)
  maxdepth <- sample(4:30, 1)
  minsplit  <- sample(seq(5, 800, by = 5), 1)
  minbucket <- sample(seq(1, minsplit), 1)

  list(
    feature_fraction = feature_fraction,
    cp = cp,
    maxdepth = min(maxdepth, 30),
    minsplit = minsplit,
    minbucket = minbucket
  )
}

evaluar_combinacion <- function(combo, num_trees, data_train, data_eval) {
  prob_acumulada <- rep(0, nrow(data_eval))

  for (arbolito in seq(num_trees)) {
    qty_campos <- as.integer(length(campos_buenos) * combo$feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos)
    campos_random <- paste(campos_random, collapse = " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita,
      data = data_train,
      xval = 0,
      control = list(
        cp = combo$cp,
        minsplit = combo$minsplit,
        minbucket = combo$minbucket,
        maxdepth = combo$maxdepth
      )
    )

    pred <- predict(modelo, data_eval, type = "prob")
    prob_acumulada <- prob_acumulada + pred[, "BAJA+2"]
  }

  prob_acumulada
}

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 100043
PARAM$num_trees_final    <- 32
PARAM$num_trees_busqueda <- 10
PARAM$qty_combinaciones  <- 50
PARAM$qty_confirmar      <- 5

## Paso 5: etapa A, exploracion aleatoria con checkpoint

In [ ]:
archivo_checkpoint_A <- "GridSearch_EtapaA_checkpoint.csv"

if (file.exists(archivo_checkpoint_A)) {
  tb_resultados_A <- fread(archivo_checkpoint_A)
  desde <- nrow(tb_resultados_A) + 1
  cat("retomando desde", desde, "\n")
} else {
  tb_resultados_A <- data.table(
    combinacion_id = integer(), feature_fraction = numeric(), cp = numeric(),
    maxdepth = integer(), minsplit = integer(), minbucket = integer(),
    auc = numeric(), tiempo_seg = numeric()
  )
  desde <- 1
}

set.seed(PARAM$semilla_primigenia)

if (desde <= PARAM$qty_combinaciones) {
  for (i in seq(desde, PARAM$qty_combinaciones)) {

    combo <- generar_combinacion()
    t0 <- Sys.time()

    prob <- evaluar_combinacion(combo, PARAM$num_trees_busqueda, dtrain_fit, dvalid)
    auc_combo <- as.numeric(pROC::auc(dvalid$clase_binaria, prob, quiet = TRUE))
    tiempo <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

    tb_resultados_A <- rbind(tb_resultados_A, data.table(
      combinacion_id = i, feature_fraction = combo$feature_fraction, cp = combo$cp,
      maxdepth = combo$maxdepth, minsplit = combo$minsplit, minbucket = combo$minbucket,
      auc = auc_combo, tiempo_seg = tiempo
    ))

    fwrite(tb_resultados_A, archivo_checkpoint_A)

    cat(sprintf("[A %d/%d] AUC=%.4f ff=%.2f cp=%s maxdepth=%d minsplit=%d minbucket=%d (%.1fs)\n",
        i, PARAM$qty_combinaciones, auc_combo, combo$feature_fraction, as.character(combo$cp),
        combo$maxdepth, combo$minsplit, combo$minbucket, tiempo))
  }
} else {
  cat("etapa A completa\n")
}

## Paso 6: etapa B, confirmacion con 32 arboles

In [ ]:
candidatas <- tb_resultados_A[order(-auc)][1:PARAM$qty_confirmar]

archivo_checkpoint_B <- "GridSearch_EtapaB_checkpoint.csv"

if (file.exists(archivo_checkpoint_B)) {
  tb_resultados_B <- fread(archivo_checkpoint_B)
} else {
  tb_resultados_B <- data.table(
    combinacion_id = integer(), feature_fraction = numeric(), cp = numeric(),
    maxdepth = integer(), minsplit = integer(), minbucket = integer(),
    auc_32 = numeric(), tiempo_seg = numeric()
  )
}

ya_evaluadas <- tb_resultados_B$combinacion_id

for (fila in seq_len(nrow(candidatas))) {
  combo_id <- candidatas$combinacion_id[fila]
  if (combo_id %in% ya_evaluadas) next

  combo <- list(
    feature_fraction = candidatas$feature_fraction[fila],
    cp = candidatas$cp[fila],
    maxdepth = candidatas$maxdepth[fila],
    minsplit = candidatas$minsplit[fila],
    minbucket = candidatas$minbucket[fila]
  )

  t0 <- Sys.time()
  prob <- evaluar_combinacion(combo, PARAM$num_trees_final, dtrain_fit, dvalid)
  auc_32 <- as.numeric(pROC::auc(dvalid$clase_binaria, prob, quiet = TRUE))
  tiempo <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

  tb_resultados_B <- rbind(tb_resultados_B, data.table(
    combinacion_id = combo_id, feature_fraction = combo$feature_fraction, cp = combo$cp,
    maxdepth = combo$maxdepth, minsplit = combo$minsplit, minbucket = combo$minbucket,
    auc_32 = auc_32, tiempo_seg = tiempo
  ))

  fwrite(tb_resultados_B, archivo_checkpoint_B)

  cat(sprintf("[B combo %d] AUC=%.4f (%.1fs)\n", combo_id, auc_32, tiempo))
}

mejor <- tb_resultados_B[which.max(auc_32)]
print(mejor)

## Paso 7: reentreno final y submit a Kaggle

In [ ]:
PARAM$rpart_final <- list(
  cp = mejor$cp,
  minsplit = mejor$minsplit,
  minbucket = mejor$minbucket,
  maxdepth = min(mejor$maxdepth, 30)
)

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob_acumulada := 0]

set.seed(PARAM$semilla_primigenia)

for (arbolito in seq(PARAM$num_trees_final)) {
  qty_campos <- as.integer(length(campos_buenos) * mejor$feature_fraction)
  campos_random <- sample(campos_buenos, qty_campos)
  campos_random <- paste(campos_random, collapse = " + ")
  formulita <- paste0("clase_ternaria ~ ", campos_random)

  modelo <- rpart(formulita,
    data = dtrain,
    xval = 0,
    control = PARAM$rpart_final
  )

  pred <- predict(modelo, dfuture, type = "prob")
  tb_prediccion[, prob_acumulada := prob_acumulada + pred[, "BAJA+2"]]
}

umbral_corte <- (1 / 40) * PARAM$num_trees_final
tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

archivo_kaggle <- "KA421_GridSearch_ArbolesAzarosos.csv"
fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

comando <- "kaggle competitions submit"
competencia <- "-c utn-2026-inicial"
arch <- paste("-f", archivo_kaggle)
mensaje <- paste0(
  "-m 'GridSearch ArbolesAzarosos 32arb ff=", mejor$feature_fraction,
  " cp=", mejor$cp, " maxdepth=", mejor$maxdepth,
  " minsplit=", mejor$minsplit, " minbucket=", mejor$minbucket, "'"
)
linea <- paste(comando, competencia, arch, mensaje)
salida <- system(linea, intern = TRUE)
cat(salida)

## Paso 8: resumen para la Planilla, hoja C4

In [ ]:
resumen_planilla <- data.table(
  integrante = "Sebastian Bianchini",
  fecha = format(Sys.Date(), "%Y-%m-%d"),
  feature_fraction = mejor$feature_fraction,
  cp = mejor$cp,
  maxdepth = mejor$maxdepth,
  minsplit = mejor$minsplit,
  minbucket = mejor$minbucket,
  num_arboles = PARAM$num_trees_final,
  auc_local_holdout = mejor$auc_32,
  qty_combinaciones_exploradas = PARAM$qty_combinaciones,
  qty_confirmadas_32arb = PARAM$qty_confirmar,
  semilla = PARAM$semilla_primigenia,
  ganancia_kaggle = NA
)

print(resumen_planilla)
fwrite(resumen_planilla, "resumen_C4_GridSearch_ArbolesAzarosos.csv")

print(tb_resultados_A[order(-auc)][1:10])